# Predicting Smartphone Addiction - Zenith Tri-Seed 120-Model GPU-Accelerated SOTA Ensemble

## Overview
This notebook implements the **Zenith Tri-Seed 10-Fold 4-Model GPU-Accelerated SOTA Ensemble (120 Deep Models)**.

### Key Architectural Features & Innovations:
1. **Hardware Acceleration**: Leveraging NVIDIA RTX 4050 GPU for XGBoost Hist (`device='cuda'`) and CatBoost (`task_type='GPU'`) alongside 16-threaded CPU LightGBM and HistGBM.
2. **Multi-Frequency Trigonometric Lookups**: High-frequency harmonic transformations (`sin` and `cos` with periods 10, 20, 50, 100) on synthetic lookup keys (`notifications_per_day`, `app_opens_per_day`, `age`).
3. **Circular Decimal Lattice & Remainder Polar Coordinates**: Sub-unit fractional remainders (`frac_col`), first decimal digits (`d1_col`), integer / half-integer indicators, and polar remainder coordinates (`sin_frac`, `cos_frac`).
4. **Transductive Population Frequency Statistics**: Empirical density features across all 987,671 samples (train + test) without target leakage.
5. **10-Fold In-Fold Bayesian Target Encoding**: Nested 10-fold inner Bayesian smoothed target statistics (`SMOOTH=10.0`) with explicit `__missing__` level preservation.
6. **Tri-Seed 120-Model Bagged Ensemble**: Combining 120 deep models across Seeds `42`, `2024`, `7` with Nelder-Mead probability meta-weights.

In [ ]:
import os
import sys
import time
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, roc_curve
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings('ignore')
print('Environment and libraries initialized successfully.')

In [ ]:
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

TARGET = 'addicted_label'
CATS = ['gender', 'stress_level', 'academic_work_impact']
NUMS = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
        'work_study_hours', 'sleep_hours', 'notifications_per_day',
        'app_opens_per_day', 'weekend_screen_time']
ALL_RAW = NUMS + CATS
FRAC_COLS = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
             'work_study_hours', 'sleep_hours', 'weekend_screen_time']
LOOKUP_COLS = ['notifications_per_day', 'app_opens_per_day', 'age']

y = train_df[TARGET].values
print(f'Training shape: {train_df.shape}, Test shape: {test_df.shape}, Target rate: {y.mean():.4f}')

In [ ]:
def engineer_base_features(df):
    d = df.copy()
    d['gender_num'] = d['gender'].map({'Male': 0, 'Female': 1, 'Other': 2}).fillna(0)
    d['stress_num'] = d['stress_level'].map({'Low': 0, 'Medium': 1, 'High': 2}).fillna(1)
    d['impact_num'] = d['academic_work_impact'].map({'Low': 0, 'Medium': 1, 'High': 2}).fillna(1)
    
    d['entertainment_time'] = d['social_media_hours'] + d['gaming_hours']
    d['screen_minus_ent'] = d['daily_screen_time_hours'] - d['entertainment_time']
    d['ent_ratio'] = d['entertainment_time'] / (d['daily_screen_time_hours'] + 1e-4)
    d['social_ratio'] = d['social_media_hours'] / (d['daily_screen_time_hours'] + 1e-4)
    d['gaming_ratio'] = d['gaming_hours'] / (d['daily_screen_time_hours'] + 1e-4)
    d['work_ratio'] = d['work_study_hours'] / (d['daily_screen_time_hours'] + 1e-4)
    d['screen_to_sleep'] = d['daily_screen_time_hours'] / (d['sleep_hours'] + 1e-4)
    d['ent_to_sleep'] = d['entertainment_time'] / (d['sleep_hours'] + 1e-4)
    d['notif_per_hour'] = d['notifications_per_day'] / (d['daily_screen_time_hours'] + 1e-4)
    d['opens_per_hour'] = d['app_opens_per_day'] / (d['daily_screen_time_hours'] + 1e-4)
    d['notif_per_open'] = d['notifications_per_day'] / (d['app_opens_per_day'] + 1e-4)
    d['weekend_vs_daily'] = d['weekend_screen_time'] - d['daily_screen_time_hours']
    d['weekend_to_daily_ratio'] = d['weekend_screen_time'] / (d['daily_screen_time_hours'] + 1e-4)
    
    d['total_accounted_time'] = d['daily_screen_time_hours'] + d['work_study_hours'] + d['sleep_hours']
    d['unaccounted_time'] = 24.0 - d['total_accounted_time']
    d['sleep_deprivation_index'] = np.maximum(0, 8.0 - d['sleep_hours'])
    d['heavy_screen_flag'] = (d['daily_screen_time_hours'] > 8.0).astype(float)
    d['heavy_social_flag'] = (d['social_media_hours'] > 4.0).astype(float)
    d['heavy_gaming_flag'] = (d['gaming_hours'] > 3.0).astype(float)
    d['heavy_notif_flag'] = (d['notifications_per_day'] > 100).astype(float)
    d['poor_sleep_flag'] = (d['sleep_hours'] < 6.0).astype(float)
    d['high_risk_score'] = (d['heavy_screen_flag'] + d['heavy_social_flag'] + d['heavy_gaming_flag'] + 
                            d['heavy_notif_flag'] + d['poor_sleep_flag'] + (d['stress_num'] == 2).astype(float))
    
    d['screen_x_stress'] = d['daily_screen_time_hours'] * (d['stress_num'] + 1)
    d['ent_x_stress'] = d['entertainment_time'] * (d['stress_num'] + 1)
    d['screen_x_impact'] = d['daily_screen_time_hours'] * (d['impact_num'] + 1)
    d['screen_x_age'] = d['daily_screen_time_hours'] * d['age']
    d['ent_x_age'] = d['entertainment_time'] * d['age']
    d['opens_x_notifs'] = d['app_opens_per_day'] * d['notifications_per_day']
    d['ent_density'] = d['entertainment_time'] / (d['sleep_hours'] + d['work_study_hours'] + 1e-4)
    d['stress_plus_impact'] = d['stress_num'] + d['impact_num']
    d['stress_x_impact'] = d['stress_num'] * d['impact_num']
    d['notif_x_stress'] = d['notifications_per_day'] * (d['stress_num'] + 1)
    d['opens_x_stress'] = d['app_opens_per_day'] * (d['stress_num'] + 1)
    d['log_daily_screen'] = np.log1p(np.maximum(0, d['daily_screen_time_hours']))
    d['log_social_media'] = np.log1p(np.maximum(0, d['social_media_hours']))
    d['log_gaming'] = np.log1p(np.maximum(0, d['gaming_hours']))
    d['log_notifications'] = np.log1p(np.maximum(0, d['notifications_per_day']))
    d['log_app_opens'] = np.log1p(np.maximum(0, d['app_opens_per_day']))
    
    for col in FRAC_COLS:
        val = np.maximum(0, d[col])
        frac = val - np.floor(val)
        d[f'{col}_frac'] = frac
        d[f'{col}_d1'] = np.floor(frac * 10.0) % 10.0
        d[f'{col}_is_int'] = (frac < 1e-5).astype(float)
        d[f'{col}_is_half'] = (np.abs(frac - 0.5) < 1e-5).astype(float)
        d[f'{col}_sin_frac'] = np.sin(2 * np.pi * frac)
        d[f'{col}_cos_frac'] = np.cos(2 * np.pi * frac)
        
    for col in LOOKUP_COLS:
        val = d[col].astype(float)
        for period in [10.0, 20.0, 50.0, 100.0]:
            d[f'{col}_sin_p{int(period)}'] = np.sin(2 * np.pi * val / period)
            d[f'{col}_cos_p{int(period)}'] = np.cos(2 * np.pi * val / period)
            
    return d

train_fe = engineer_base_features(train_df)
test_fe = engineer_base_features(test_df)

full_df = pd.concat([train_df[ALL_RAW], test_df[ALL_RAW]], axis=0)
for col in ALL_RAW:
    freq_map = full_df[col].value_counts(normalize=True).to_dict()
    train_fe[f'{col}_transductive_freq'] = train_df[col].map(freq_map).fillna(0).values.astype(np.float32)
    test_fe[f'{col}_transductive_freq'] = test_df[col].map(freq_map).fillna(0).values.astype(np.float32)

base_feature_cols = [c for c in train_fe.columns if c not in ['id', TARGET, 'gender', 'stress_level', 'academic_work_impact']]
print(f'Total base engineered features: {len(base_feature_cols)}')

In [ ]:
def build_all_te(trn_df, trn_y, val_df, tst_df, cols=ALL_RAW, n_splits=10, smooth=10.0, seed=42):
    trn_out = pd.DataFrame(index=trn_df.index)
    val_out = pd.DataFrame(index=val_df.index)
    tst_out = pd.DataFrame(index=tst_df.index)
    
    global_mean = trn_y.mean()
    inner_skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    for col in cols:
        te_trn = np.zeros(len(trn_df), dtype=np.float32)
        for in_trn_idx, in_val_idx in inner_skf.split(trn_df, trn_y):
            sub_df = trn_df.iloc[in_trn_idx]
            sub_y = trn_y.iloc[in_trn_idx]
            stats = sub_y.groupby(sub_df[col]).agg(['count', 'mean'])
            smooth_te = (stats['count'] * stats['mean'] + smooth * global_mean) / (stats['count'] + smooth)
            te_dict = smooth_te.to_dict()
            te_trn[in_val_idx] = trn_df[col].iloc[in_val_idx].map(te_dict).fillna(global_mean).values
            
        full_stats = trn_y.groupby(trn_df[col]).agg(['count', 'mean'])
        smooth_full = (full_stats['count'] * full_stats['mean'] + smooth * global_mean) / (full_stats['count'] + smooth)
        full_dict = smooth_full.to_dict()
        
        trn_out[f'{col}_te'] = te_trn
        val_out[f'{col}_te'] = val_df[col].map(full_dict).fillna(global_mean).values.astype(np.float32)
        tst_out[f'{col}_te'] = tst_df[col].map(full_dict).fillna(global_mean).values.astype(np.float32)
        
        freq = trn_df[col].value_counts(normalize=True).to_dict()
        trn_out[f'{col}_freq'] = trn_df[col].map(freq).fillna(0).values.astype(np.float32)
        val_out[f'{col}_freq'] = val_df[col].map(freq).fillna(0).values.astype(np.float32)
        tst_out[f'{col}_freq'] = tst_df[col].map(freq).fillna(0).values.astype(np.float32)
        
    return trn_out, val_out, tst_out

print('10-Fold In-Fold Bayesian Target Encoder ready.')

In [ ]:
N_SPLITS = 10
SEEDS = [42, 2024, 7]

oof_lgb_total = np.zeros(len(train_df))
oof_xgb_total = np.zeros(len(train_df))
oof_cat_total = np.zeros(len(train_df))
oof_hgb_total = np.zeros(len(train_df))

test_preds_lgb = np.zeros(len(test_df))
test_preds_xgb = np.zeros(len(test_df))
test_preds_cat = np.zeros(len(test_df))
test_preds_hgb = np.zeros(len(test_df))

for s_idx, seed in enumerate(SEEDS):
    print(f'\n========== STARTING SEED {seed} ({s_idx+1}/{len(SEEDS)}) ==========')
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    oof_lgb_seed = np.zeros(len(train_df))
    oof_xgb_seed = np.zeros(len(train_df))
    oof_cat_seed = np.zeros(len(train_df))
    oof_hgb_seed = np.zeros(len(train_df))
    
    for fold, (trn_idx, val_idx) in enumerate(skf.split(train_fe, y)):
        trn_raw = train_df.iloc[trn_idx]
        val_raw = train_df.iloc[val_idx]
        trn_y = train_df[TARGET].iloc[trn_idx]
        val_y = train_df[TARGET].iloc[val_idx]
        
        trn_te, val_te, tst_te = build_all_te(trn_raw, trn_y, val_raw, test_df, cols=ALL_RAW, n_splits=10, smooth=10.0, seed=seed+fold)
        
        X_trn = pd.concat([train_fe.iloc[trn_idx][base_feature_cols], trn_te], axis=1)
        X_val = pd.concat([train_fe.iloc[val_idx][base_feature_cols], val_te], axis=1)
        X_tst = pd.concat([test_fe[base_feature_cols], tst_te], axis=1)
        
        # 1. Deep LightGBM
        m_lgb = lgb.LGBMClassifier(
            n_estimators=1700, learning_rate=0.025, num_leaves=190, max_depth=12,
            min_child_samples=40, subsample=0.80, subsample_freq=1, colsample_bytree=0.60,
            reg_alpha=0.20, reg_lambda=2.0, random_state=seed+fold, n_jobs=-1, verbose=-1
        )
        m_lgb.fit(X_trn, trn_y, eval_set=[(X_val, val_y)], callbacks=[lgb.early_stopping(50, verbose=False)])
        p_val_lgb = m_lgb.predict_proba(X_val)[:, 1]
        oof_lgb_seed[val_idx] = p_val_lgb
        test_preds_lgb += m_lgb.predict_proba(X_tst)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # 2. Deep XGBoost Hist on GPU
        m_xgb = xgb.XGBClassifier(
            n_estimators=1400, learning_rate=0.030, max_depth=8, min_child_weight=35,
            subsample=0.80, colsample_bytree=0.60, reg_alpha=0.20, reg_lambda=2.0,
            tree_method='hist', device='cuda', random_state=seed+fold, verbosity=0
        )
        m_xgb.fit(X_trn, trn_y, eval_set=[(X_val, val_y)], verbose=False)
        p_val_xgb = m_xgb.predict_proba(X_val)[:, 1]
        oof_xgb_seed[val_idx] = p_val_xgb
        test_preds_xgb += m_xgb.predict_proba(X_tst)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # 3. Deep CatBoost on GPU
        m_cat = CatBoostClassifier(
            iterations=1600, learning_rate=0.035, depth=7, l2_leaf_reg=3.0,
            task_type='GPU', random_seed=seed+fold, verbose=False
        )
        m_cat.fit(X_trn, trn_y, eval_set=(X_val, val_y), early_stopping_rounds=50, verbose=False)
        p_val_cat = m_cat.predict_proba(X_val)[:, 1]
        oof_cat_seed[val_idx] = p_val_cat
        test_preds_cat += m_cat.predict_proba(X_tst)[:, 1] / (N_SPLITS * len(SEEDS))
        
        # 4. HistGradientBoosting
        m_hgb = HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.035, max_leaf_nodes=190,
            min_samples_leaf=40, l2_regularization=0.5, random_state=seed+fold
        )
        m_hgb.fit(X_trn, trn_y)
        p_val_hgb = m_hgb.predict_proba(X_val)[:, 1]
        oof_hgb_seed[val_idx] = p_val_hgb
        test_preds_hgb += m_hgb.predict_proba(X_tst)[:, 1] / (N_SPLITS * len(SEEDS))
        
        print(f'Fold {fold+1} Finished | LGB={roc_auc_score(val_y, p_val_lgb):.5f}, XGB={roc_auc_score(val_y, p_val_xgb):.5f}, CAT={roc_auc_score(val_y, p_val_cat):.5f}')
        
    oof_lgb_total += oof_lgb_seed / len(SEEDS)
    oof_xgb_total += oof_xgb_seed / len(SEEDS)
    oof_cat_total += oof_cat_seed / len(SEEDS)
    oof_hgb_total += oof_hgb_seed / len(SEEDS)

print('\nAll 120 models trained successfully.')

In [ ]:
print('Optimizing non-negative ensemble weights...')
def loss_func(w):
    w = np.maximum(0, w)
    if np.sum(w) == 0: return 1.0
    w = w / np.sum(w)
    p = w[0]*oof_lgb_total + w[1]*oof_xgb_total + w[2]*oof_cat_total + w[3]*oof_hgb_total
    return -roc_auc_score(y, p)

res = minimize(loss_func, [0.38, 0.46, 0.16, 0.00], method='Nelder-Mead')
opt_w = np.maximum(0, res.x) / np.sum(np.maximum(0, res.x))

w_lgb, w_xgb, w_cat, w_hgb = opt_w[0], opt_w[1], opt_w[2], opt_w[3]
print(f'Optimal Weights: LightGBM = {w_lgb:.4f}, XGBoost = {w_xgb:.4f}, CatBoost = {w_cat:.4f}, HistGBM = {w_hgb:.4f}')

final_oof_probs = w_lgb * oof_lgb_total + w_xgb * oof_xgb_total + w_cat * oof_cat_total + w_hgb * oof_hgb_total
final_test_preds = w_lgb * test_preds_lgb + w_xgb * test_preds_xgb + w_cat * test_preds_cat + w_hgb * test_preds_hgb

final_oof_auc = roc_auc_score(y, final_oof_probs)
final_preds_binary = (final_oof_probs >= 0.50).astype(int)
acc = accuracy_score(y, final_preds_binary)
f1 = f1_score(y, final_preds_binary)
prec = precision_score(y, final_preds_binary)
rec = recall_score(y, final_preds_binary)

print('=' * 80)
print(f'ZENITH TRI-SEED 120-MODEL ENSEMBLE OOF ROC-AUC: {final_oof_auc:.5f}')
print(f'Accuracy:  {acc*100:.2f}%')
print(f'F1-Score:  {f1:.5f}')
print(f'Precision: {prec:.5f}')
print(f'Recall:    {rec:.5f}')
print('=' * 80)

In [ ]:
sub = pd.DataFrame({'id': test_df['id'].values, TARGET: final_test_preds})
sub.to_csv('submission.csv', index=False)
print(f'submission.csv successfully saved: {len(sub)} samples, mean prob: {final_test_preds.mean():.6f}')